In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

# Create a single DEM image and clip it
elevation = (
    dataset
    .select('DEM')
    .first()
    .clip(panama_geom)
)

# ~11,131m resolution
temp = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterDate("2023-01-01", "2023-02-01")
    .first()
    .clip(panama_geom)
)

temp_reprojected = temp.reproject(
    crs = elevation.projection(), scale = temp.projection().nominalScale()
)

Map.addLayer(
    temp,
    {
        "bands": ["temperature_2m"],
        "min": 250,
        "max": 320
    },
    "Temp"
)

Map.addLayer(
    temp_reprojected,
    {
        "bands": ["temperature_2m"],
        "min": 250,
        "max": 320
    },
    "Temp Reprojected"
)

Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…